# 04 — Video Processor

Renders moving clips, subtitles, optional music, then a black source card. It validates final duration before marking the job complete.

In [ ]:
import os, sys, subprocess
ROOT="/content/black-history-factory"
REPO_URL="https://github.com/jonbBla/black-history-factory.git"
if not os.path.exists(ROOT):
    subprocess.run(["git","clone",REPO_URL,ROOT], check=True)
sys.path.insert(0, ROOT)
subprocess.run(["apt-get","update","-qq"],check=True); subprocess.run(["apt-get","install","-y","-qq","ffmpeg"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","pillow"],check=True)
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(f"[SETUP] Drive: {paths.root}")


In [ ]:
from factory.video_engine import run
from factory.thumbnail_engine import run as make_thumbnail
from factory.utils import read_json,write_json_atomic
from factory import status
def process_one():
    jobs=[]
    for jid in sorted(os.listdir(paths("02_JOBS"))):
        m=read_json(paths.manifest(jid),{}) or {}
        if m.get("status")=="AUDIO_READY": jobs.append(jid)
    if not jobs:
        print("[VIDEO] No audio-ready job."); return False
    jid=jobs[0]; scenes=read_json(paths.scenes(jid),[]); research=read_json(paths.verified(jid),{})
    def progress(n,total): status.set_processor(paths,"video","running",jid,"rendering",f"scene {n}/{total}",n,total)
    status.set_processor(paths,"video","running",jid,"rendering","starting",0,len(scenes))
    try:
        final,seconds=run(paths,jid,scenes,research,config,progress)
        try: make_thumbnail(paths,jid,os.path.join(paths.images_dir(jid),"scene_001.png"),(read_json(paths.manifest(jid),{}) or {}).get("title",""))
        except Exception as e: print(f"[VIDEO] Thumbnail warning: {e}")
        m=read_json(paths.manifest(jid),{}) or {}; m.update(status="COMPLETED",final_video=final,final_seconds=seconds); write_json_atomic(paths.manifest(jid),m)
        status.append_history(paths,jid,m.get("title",jid),final,paths.thumbnail(jid),seconds)
        status.set_processor(paths,"video","idle",jid,"completed",f"{seconds:.1f}s final",1,1); print(f"[VIDEO] COMPLETE {jid} | {seconds:.1f}s | {final}"); return True
    except Exception as e:
        m=read_json(paths.manifest(jid),{}) or {}; m.update(status="VIDEO_ERROR",video_error=str(e)); write_json_atomic(paths.manifest(jid),m)
        status.set_processor(paths,"video","error",jid,"failed",str(e)); print(f"[VIDEO] ERROR {jid} | {e}"); return False
process_one()


In [ ]:
# Optional after the one-video test.
while process_one(): pass
